"""
XGBoost + Optuna 하이퍼파라미터 최적화
- 평가지표: RMSE
- 교차검증: 5-Fold
- calories_burned_eda_ver2 전처리 로직 포함
"""

In [32]:
import pandas as pd
import numpy as np
import random
import os
import warnings
warnings.filterwarnings('ignore')

import optuna
from optuna.samplers import TPESampler
import xgboost as xgb
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor

In [22]:
# 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [23]:
#데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

Train shape: (7500, 11)
Test shape:  (7500, 10)


피처엔지니어링

In [41]:
# Feature Engineering
def feature_engineering(df):
    df = df.copy()
    
    # ── 기본 변환 ──
    # Height 통합 (Feet → Inches)
    # df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']

    # # BMI 계산: weight(lb) / height(inches)^2 * 703
    # df['BMI'] = (df['Weight(lb)'] / (df['Height_Total_Inches'] ** 2)) * 703
    # df['BMI_x_Duration'] = df['BMI'] * df['Exercise_Duration']

    # Duration_bin: 운동 시간 구간화 
    # 단시간/중간/장시간 운동의 비선형 패턴 포착
    df['Duration_bin'] = pd.cut(
        df['Exercise_Duration'],
        bins=[0, 10, 20, 30],
        labels=[0, 1, 2]  # 0=단시간, 1=중간, 2=장시간
    ).astype(int)

    #  Temp_diff: 정상 체온(98.6°F) 대비 상승폭 
    # 운동 강도를 직접 반영하는 지표
    df["Temp_diff"] = df["Body_Temperature(F)"] - 98.6
    df["Dur_x_TempDiff"] = df["Exercise_Duration"] * df["Temp_diff"]
    
    # Interaction Features (상관계수 0.91~0.97로 매우 유용)
    df['Duration_x_BPM'] = df['Exercise_Duration'] * df['BPM']
    # df['BPM_x_Temp'] = df['BPM'] * df['Body_Temperature(F)']
    df['Duration_sq'] = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq'] = df['Temp_diff'] ** 2


    # 3-way interaction (최상위 3개 변수 결합) 
    df['Dur_BPM_Temp'] = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']

    # 범주형 인코딩
    # Gender: M=1, F=0
    df['Gender_encoded'] = (df['Gender'] == 'M').astype(int)


    return df

#train test 피처 일치해야함
train_feature = feature_engineering(train)
test_feature = feature_engineering(test)

missing_in_test = set(train_feature.columns) - set(test_feature.columns)
missing_in_train = set(test_feature.columns) - set(train_feature.columns)

print("missing_in_test:", missing_in_test)
print("missing_in_train:", missing_in_train)

missing_in_test: {'Calories_Burned'}
missing_in_train: set()


학습 데이터 분리

In [42]:
# X에서 제외할 컬럼
drop_cols = ["Calories_Burned", "Gender", "Weight_Status", "ID"]

# X 생성
X = train_feature.drop(columns=drop_cols) # X에서는 타겟 제거
y = train_feature["Calories_Burned"] # 타겟 따로 분리
# test에는 원래 타겟이 없음
X_test = test_feature.drop(columns=["Gender", "Weight_Status", "ID"])

# 실제 사용하는 피처 목록
feature_cols = X.columns.tolist()

print(f"\n피처 수: {len(feature_cols)}")
print(f"피처 목록: {feature_cols}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

# train test 컬럼 동일한지 체크
print(X.columns.equals(X_test.columns))
#문자열 남아있는지 체크
print(X.dtypes[X.dtypes == "object"])



피처 수: 15
피처 목록: ['Exercise_Duration', 'Body_Temperature(F)', 'BPM', 'Height(Feet)', 'Height(Remainder_Inches)', 'Weight(lb)', 'Age', 'Duration_bin', 'Temp_diff', 'Dur_x_TempDiff', 'Duration_x_BPM', 'Duration_sq', 'Temp_diff_sq', 'Dur_BPM_Temp', 'Gender_encoded']
X shape: (7500, 15), y shape: (7500,)
True
Series([], dtype: object)


Optuna 목적함수 정의

In [ ]:
def objective(trial):
    """Optuna objective: 5-Fold CV RMSE 최소화"""

    params = {
        'n_estimators':      trial.suggest_int('n_estimators',1500,3000),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.03, 0.15),
        'subsample':         trial.suggest_float('subsample', 0.6, 0.8),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 6.0, log=True),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0)
    }

    model = xgb.XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        verbosity=0,
    )

    # 5-Fold Cross Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    rmse_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )

        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_scores.append(rmse)

    mean_rmse = np.mean(rmse_scores)
    return mean_rmse

In [29]:
#Optuna 최적화 실행
print("\n" + "=" * 70)
print("  Optuna XGBoost 하이퍼파라미터 최적화 시작")
print("  평가지표: RMSE | 교차검증: 5-Fold")
print("=" * 70)

# Optuna study 생성
sampler = TPESampler(seed=42)
study = optuna.create_study(
    direction='minimize',  # RMSE 최소화
    sampler=sampler,
    study_name='xgboost_calories_prediction'
)

# 최적화 실행 (100 trials)
study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True,
    callbacks=[lambda study, trial: print(
        f"  Trial {trial.number:3d} | RMSE: {trial.value:.4f} | "
        f"Best: {study.best_value:.4f}"
    ) if trial.number % 10 == 0 else None]
)

# 최적 결과 출력
print("\n" + "=" * 70)
print("  최적화 결과")
print("=" * 70)

best_params = study.best_params
best_rmse = study.best_value

print(f"\n  Best RMSE (5-Fold CV): {best_rmse:.6f}")
print(f"\n  최적 하이퍼파라미터:")
for param, value in best_params.items():
    if isinstance(value, float):
        print(f"    {param:25s}: {value:.6f}")
    else:
        print(f"    {param:25s}: {value}")


[I 2026-02-12 19:04:19,821] A new study created in memory with name: xgboost_calories_prediction



  Optuna XGBoost 하이퍼파라미터 최적화 시작
  평가지표: RMSE | 교차검증: 5-Fold


Best trial: 0. Best value: 3.10982:   2%|▏         | 1/50 [00:08<06:53,  8.45s/it]

[I 2026-02-12 19:04:28,269] Trial 0 finished with value: 3.10982294761909 and parameters: {'n_estimators': 2062, 'max_depth': 10, 'learning_rate': 0.1178392730173686, 'subsample': 0.7197316968394073, 'colsample_bytree': 0.5468055921327309, 'min_child_weight': 4, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.4012593978442727, 'gamma': 3.005575058716044}. Best is trial 0 with value: 3.10982294761909.
  Trial   0 | RMSE: 3.1098 | Best: 3.1098


Best trial: 1. Best value: 1.85462:   4%|▍         | 2/50 [00:19<07:45,  9.70s/it]

...

[I 2026-02-12 19:17:31,876] Trial 47 finished with value: 1.9231277738028045 and parameters: {'n_estimators': 2573, 'max_depth': 5, 'learning_rate': 0.12488766379644967, 'subsample': 0.6954370153054814, 'colsample_bytree': 0.524199026173558, 'min_child_weight': 4, 'reg_alpha': 0.021313572296067475, 'reg_lambda': 0.001240587198941417, 'gamma': 0.28400377221003636}. Best is trial 33 with value: 1.472051661236043.


Best trial: 33. Best value: 1.47205:  98%|█████████▊| 49/50 [13:29<00:16, 16.36s/it]

[I 2026-02-12 19:17:49,350] Trial 48 finished with value: 1.54061762815614 and parameters: {'n_estimators': 2514, 'max_depth': 4, 'learning_rate': 0.05714402958293317, 'subsample': 0.6408165238780114, 'colsample_bytree': 0.5852071433664475, 'min_child_weight': 16, 'reg_alpha': 0.7888866235456231, 'reg_lambda': 6.98884400081982e-05, 'gamma': 0.4437828719834319}. Best is trial 33 with value: 1.472051661236043.


Best trial: 33. Best value: 1.47205: 100%|██████████| 50/50 [13:42<00:00, 16.45s/it]

[I 2026-02-12 19:18:02,431] Trial 49 finished with value: 1.7913450153247628 and parameters: {'n_estimators': 2671, 'max_depth': 5, 'learning_rate': 0.0426705279431358, 'subsample': 0.7692724256056398, 'colsample_bytree': 0.6136658811591361, 'min_child_weight': 10, 'reg_alpha': 2.7758516377838705, 'reg_lambda': 0.0009288550399963784, 'gamma': 2.7703935930209482}. Best is trial 33 with value: 1.472051661236043.

  최적화 결과

  Best RMSE (5-Fold CV): 1.472052

  최적 하이퍼파라미터:
    n_estimators             : 2919
    max_depth                : 4
    learning_rate            : 0.048668
    subsample                : 0.674018
    colsample_bytree         : 0.608423
    min_child_weight         : 12
    reg_alpha                : 0.000000
    reg_lambda               : 0.001195
    gamma                    : 0.002848


In [33]:
y_log = np.log1p(y)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []

for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]

    model = LGBMRegressor(random_state=42)
    model.fit(X_tr, y_tr)

    pred_log = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred_log))
    rmse_scores.append(rmse)

print("Log-scale RMSE:", np.mean(rmse_scores))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000426 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1956
[LightGBM] [Info] Number of data points in the train set: 6000, number of used features: 19
[LightGBM] [Info] Start training from score 4.154307
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000457 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6000, number of used features: 19
[LightGBM] [Info] Start training from score 4.163583
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1956
[LightGBM] [Info] Number of data points in the train set: 6000, number of used features: 19
[LightGBM] [Info] Start traini

In [35]:
rmse_original = 1.37  # 기존 마지막 시도했던 CV 값
y_range = y.max() - y.min()
y_mean = y.mean()

print("RMSE / range:", rmse_original / y_range)
print("RMSE / mean :", rmse_original / y_mean)

RMSE / range: 0.004581939799331104
RMSE / mean : 0.015328934314584045


In [ ]:
# 최적 파라미터로 최종 모델 학습 + Fold별 성능

print("\n" + "=" * 70)
print("  최적 파라미터로 5-Fold CV 상세 결과")
print("=" * 70)

final_model = xgb.XGBRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    verbosity=0,
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmse_list = []
fold_models = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBRegressor(
        **best_params,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        verbosity=0,
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    fold_rmse_list.append(rmse)
    fold_models.append(model)

    print(f"  Fold {fold_idx + 1} | RMSE: {rmse:.6f}")

mean_rmse = np.mean(fold_rmse_list)
std_rmse = np.std(fold_rmse_list)

print(f"\n  {'─' * 40}")
print(f"  평균 RMSE : {mean_rmse:.6f}")
print(f"  표준편차  : {std_rmse:.6f}")
print(f"  범위      : {min(fold_rmse_list):.6f} ~ {max(fold_rmse_list):.6f}")




  최적 파라미터로 5-Fold CV 상세 결과
  Fold 1 | RMSE: 1.776143
  Fold 2 | RMSE: 1.400502
  Fold 3 | RMSE: 1.345688
  Fold 4 | RMSE: 1.418482
  Fold 5 | RMSE: 1.419443

  ────────────────────────────────────────
  평균 RMSE : 1.472052
  표준편차  : 0.154400
  범위      : 1.345688 ~ 1.776143


TypeError: lightgbm.sklearn.LGBMRegressor() got multiple values for keyword argument 'n_estimators'

In [48]:
# 전체 피처 버전 X, y 재정의
drop_cols = ["Calories_Burned", "Gender", "Weight_Status", "ID"]
X = train_feature.drop(columns=[c for c in drop_cols if c in train_feature.columns])
y = train_feature["Calories_Burned"]
print(X.dtypes[X.dtypes == "object"])
print(X.columns.tolist()) 

Series([], dtype: object)
['Exercise_Duration', 'Body_Temperature(F)', 'BPM', 'Height(Feet)', 'Height(Remainder_Inches)', 'Weight(lb)', 'Age', 'Duration_bin', 'Temp_diff', 'Dur_x_TempDiff', 'Duration_x_BPM', 'Duration_sq', 'Temp_diff_sq', 'Dur_BPM_Temp', 'Gender_encoded']


# 다시 LightGBM

In [47]:
#lightGBM 실험
from lightgbm import LGBMRegressor

print(X.shape, y.shape)
best_params_lgb = {
    "learning_rate": 0.05878700609192509,
    "num_leaves": 32,
    "max_depth": 8,
    "min_child_samples": 13,
    "subsample": 0.9252051762811434,
    "colsample_bytree": 0.9071376601278041,
    "reg_alpha": 0.0002260716835361878,
    "reg_lambda": 0.3269912384205852,
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    y_tr_log = np.log1p(y_tr)

    model = LGBMRegressor(
        random_state=42,
        n_estimators=2000,   # 너가 정한 값
        n_jobs=-1,
        verbose=-1,
        **best_params_lgb       #  LightGBM용 파라미터만!
    )

    model.fit(X_tr, y_tr_log)
    pred_log = model.predict(X_va)
    pred_kcal = np.expm1(pred_log)

    oof_pred[va_idx] = pred_kcal

    rmse = np.sqrt(mean_squared_error(y_va, pred_kcal))
    fold_rmses.append(rmse)
    print(f"Fold {fold} RMSE(kcal): {rmse:.6f}")

oof_rmse = np.sqrt(mean_squared_error(y, oof_pred))

print("-" * 50)
print("Fold mean:", np.mean(fold_rmses))
print("Fold std :", np.std(fold_rmses))
print("OOF RMSE :", oof_rmse)

(7500, 15) (7500,)
Fold 1 RMSE(kcal): 1.806272
Fold 2 RMSE(kcal): 1.173664
Fold 3 RMSE(kcal): 1.248561
Fold 4 RMSE(kcal): 1.256065
Fold 5 RMSE(kcal): 1.287483
--------------------------------------------------
Fold mean: 1.3544091234921978
Fold std : 0.22899915753747757
OOF RMSE : 1.3736319332163829


In [49]:
def objective_lgb_refine(trial):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.06),
        "num_leaves": trial.suggest_int("num_leaves", 24, 40),
        "max_depth": trial.suggest_int("max_depth", 6, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 8, 20),

        "subsample": trial.suggest_float("subsample", 0.85, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.85, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
    }

    model = LGBMRegressor(
        random_state=42,
        n_estimators=3000,
        n_jobs=-1,
        verbose=-1,
        **params
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmses = []

    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        y_tr_log = np.log1p(y_tr)
        model.fit(X_tr, y_tr_log)

        pred = np.expm1(model.predict(X_va))
        rmse = np.sqrt(mean_squared_error(y_va, pred))
        rmses.append(rmse)

    return np.mean(rmses)

study = optuna.create_study(direction="minimize")
study.optimize(objective_lgb_refine, n_trials=40)

[I 2026-02-12 20:48:24,653] A new study created in memory with name: no-name-ac31ee18-b621-417f-9a95-f955c18c510b
[I 2026-02-12 20:48:54,958] Trial 0 finished with value: 1.8656688918064461 and parameters: {'learning_rate': 0.04385614797458565, 'num_leaves': 34, 'max_depth': 8, 'min_child_samples': 14, 'subsample': 0.9002851516354031, 'colsample_bytree': 0.9286962163443486, 'reg_alpha': 0.17393604439754712, 'reg_lambda': 0.08154650597761348}. Best is trial 0 with value: 1.8656688918064461.
[I 2026-02-12 20:49:17,175] Trial 1 finished with value: 2.2560590229441493 and parameters: {'learning_rate': 0.048911249053049576, 'num_leaves': 39, 'max_depth': 7, 'min_child_samples': 13, 'subsample': 0.8770172692912933, 'colsample_bytree': 0.9994166680348034, 'reg_alpha': 0.2987249299132127, 'reg_lambda': 0.575020856228199}. Best is trial 0 with value: 1.8656688918064461.
[I 2026-02-12 20:49:48,156] Trial 2 finished with value: 1.9191193855254383 and parameters: {'learning_rate': 0.03167432851268

# Seed Ensemble

In [ ]:
seeds = [42, 52, 62, 72, 82] 

In [ ]:
# Optuna 최적화 히스토리 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Optimization History
trials = study.trials
trial_numbers = [t.number for t in trials]
trial_values = [t.value for t in trials]
best_values = [min(trial_values[:i+1]) for i in range(len(trial_values))]

axes[0].scatter(trial_numbers, trial_values, alpha=0.4, s=20, color='gray', label='Trial RMSE')
axes[0].plot(trial_numbers, best_values, color='red', linewidth=2, label='Best RMSE')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('RMSE')
axes[0].set_title('Optuna Optimization History', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Hyperparameter Importance (상위 파라미터)
param_importances = optuna.importance.get_param_importances(study)
params = list(param_importances.keys())
importances = list(param_importances.values())

axes[1].barh(params[::-1], importances[::-1], color='teal', alpha=0.8)
axes[1].set_xlabel('Importance')
axes[1].set_title('Hyperparameter Importance', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("  optuna_history.png 저장 완료")


In [37]:
# Test 데이터 예측 (앙상블)
print("\n" + "=" * 70)
print("  Test 데이터 예측 (5-Fold 앙상블)")
print("=" * 70)

# 5개 모델의 예측 평균
test_preds = np.mean([m.predict(X_test) for m in fold_models], axis=0)

submission = pd.DataFrame({
    'ID': test['ID'],
    'Calories_Burned': test_preds
})

submission.to_csv('submission.csv', index=False)
print(f"\n  submission.csv 저장 완료!")
print(f"  예측값 통계:")
print(f"    평균: {test_preds.mean():.2f}")
print(f"    최소: {test_preds.min():.2f}")
print(f"    최대: {test_preds.max():.2f}")


  Test 데이터 예측 (5-Fold 앙상블)

  submission.csv 저장 완료!
  예측값 통계:
    평균: 89.68
    최소: -2.69
    최대: 295.33


In [ ]:
pred_log = final_model.predict(test_x)
pred = np.expm1(pred_log)
# 로그 스케일인지 확인 (평균이 0~6이면 log 의심)
print("Pred mean:", pred.mean())

# 음수 값 있는지 체크 (칼로리는 음수 불가)
print("Min pred:", pred.min())

# 비정상 범위 체크 (300 초과 과도값 확인)
print("Max pred:", pred.max())

In [ ]:
print("\n" + "=" * 70)
print("  최종 요약")
print("=" * 70)
print(f"""
모델          : XGBoost (Optuna 최적화)
탐색 Trials   : {len(study.trials)}
교차검증      : 5-Fold
평가지표      : RMSE

Best CV RMSE  : {best_rmse:.6f}
평균 CV RMSE  : {mean_rmse:.6f} ± {std_rmse:.6f}

저장 파일:
    - submission.csv        (Test 예측 결과)
    - feature_importance.png (피처 중요도)
    - optuna_history.png     (최적화 히스토리)
""")
print("=" * 70)
